In [1]:
%load_ext autoreload
%autoreload 2
import os
if not hasattr(__builtins__, '_cwd_set'):
    os.chdir('..')
    __builtins__._cwd_set = True

In [2]:
from pathlib import Path
from pprint import pprint

import httpx

In [3]:
from util import util

In [4]:
MODEL_PATH = Path('data/models/ToolCallTest.xml')
source_xml = MODEL_PATH.read_text(encoding='utf-8')
source_graph = util.import_xml(source_xml)
source_marking = util.marking(source_graph)

exported_xml = util.export_xml(source_graph)
local_roundtrip_graph = util.import_xml(exported_xml)
local_roundtrip_marking = util.marking(local_roundtrip_graph)

print('Generated XML characters:', len(exported_xml))
print('Marking difference after local round trip:')
pprint(util.marking_difference(source_marking, local_roundtrip_marking))

Generated XML characters: 6095
Marking difference after local round trip:
[]


In [5]:
source_graph

In [6]:
API_URL = 'http://127.0.0.1:8000/api/chat/response'
DCR_CHAT = 1
request_body = {
    'text': 'Starting DCR Chat session',
    'chat_type': DCR_CHAT,
    'graph_xml': exported_xml,  # Complete XML string in the JSON body.
    'dcr_role': 'Citizen'
}
print({**request_body, 'graph_xml': f'<{len(exported_xml)} XML characters>'})
execution_result = httpx.post(API_URL, json=request_body, timeout=30).json()

{'text': 'Starting DCR Chat session', 'chat_type': 1, 'graph_xml': '<6095 XML characters>', 'dcr_role': 'Citizen'}


ReadTimeout: timed out

In [ ]:
session_id = execution_result['session_id']
dcr_graph = util.import_xml(execution_result['graph_xml'])
backend_marking = util.marking(dcr_graph)

In [ ]:
HISTORY_URL = 'http://127.0.0.1:8000/api/chat/history'
hist_response = httpx.post(HISTORY_URL, json={"session_id":session_id}, timeout=30)
hist_response.json()

In [ ]:
print(execution_result['text'])

In [ ]:
answer = "I was born in 2015"
request_body = {
    'text': answer,
    'session_id': session_id,
    'act_id': execution_result['act_id'],
    'dcr_role': 'Citizen'
}
execution_result = httpx.post(API_URL, json=request_body, timeout=30).json()

## System/LLM interaction boundaries with DCR Graphs

How would dcr graphs with multiple data variables look like?

In [8]:
from pm4py.objects.dcr.ocdcr.semantics import DcrSemantics
from pm4py.objects.dcr.ocdcr import obj
from tools.summarize_case import SummarizeCaseHistory
from tools.find_similar_cases import FindSimilarCases
from tools.find_relevant_laws import FindRelevantLaws

dcr_semantics = DcrSemantics()
computation = [("source","tool")]

ed1 = obj.DcrEventData(name="age", data_type=int)
act1 = obj.DcrActivity("Event_act1",label="Age",role="Citizen", 
                       takesInput=True, 
                       eventData=ed1,
                       priority=1)

ed2 = obj.DcrEventData(name="threshold", data_type=bool)
act2 = obj.DcrActivity("Event_act2",label="Age Threshold",role="Robot", 
                       takesInput=True, 
                       eventData=ed2,
                       priority=2)
ed21 = obj.DcrEventData(name="relevant_laws", data_type=str)
act21 = obj.DcrActivity("Event_act21",label="Relevant laws",role="Robot", 
                    #    eventData=ed21,
                       priority=3, 
                       computation = [("source","tool")])
ed22 = obj.DcrEventData(name="relevant_cases", data_type=str)
act22 = obj.DcrActivity("Event_act22",label="Relevant cases",role="Robot", 
                    #    eventData=ed21,
                       priority=4, 
                       computation = [("source","tool")])

ed3 = obj.DcrEventData(name="summary", data_type=str)
act3 = obj.DcrActivity("Event_act3",label="Summarize case",role="Robot",
                    #    eventData=ed3,
                       priority=5, 
                       computation = [("source","tool","graph","executions")])
relations = {
    obj.DcrSetValue(act1, act2, [("source", "data"), ">=", 18]),
    obj.DcrConstraint(obj.RelationType.C, act2, act21, guard=[("source", "data"), "==", True]),
    obj.DcrConstraint(obj.RelationType.C, act2, act22, guard=[("source", "data"), "==", True]),
    obj.DcrConstraint(obj.RelationType.C, act1, act2),
    obj.DcrConstraint(obj.RelationType.C, act2, act3),
    obj.DcrConstraint(obj.RelationType.C, act22, act3),
    obj.DcrConstraint(obj.RelationType.C, act21, act3),
}
act21.description = "Find relevant laws about an underage child that had access to alcohol."
act22.description = "Find relevant cases where an underage child was drunk." 
act3.description = "Make a brief summary of this case!"
act21.tool_call = FindRelevantLaws().answer
act22.tool_call = FindSimilarCases().answer
act3.tool_call = SummarizeCaseHistory().get_summary

activities = {act1, act2, act21, act22, act3}
graph = obj.DcrGraph("testGraph", elements=activities,relations=relations)

In [9]:
dcr_semantics.executeActivity(obj.DcrExecution("Event_act1", 18), graph)

In [11]:
dcr_semantics.executeActivity(obj.DcrExecution("Event_act2"), graph)

In [13]:
dcr_semantics.executeActivity(obj.DcrExecution("Event_act21"), graph)

FindRelevantLaws


/home/vco/.pyenv/versions/dcrcontroller/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 13433.96it/s]


In [15]:

dcr_semantics.executeActivity(obj.DcrExecution("Event_act22"), graph)


FindSimilarCases


In [ ]:

dcr_semantics.executeActivity(obj.DcrExecution("Event_act3"), graph)

In [16]:
for e in graph.executions:
    act = graph.getActivity(e.activityID)
    print("----------------------")
    print(act.label, "\n",act.description ,"\n",act.computation,act.tool_call,"\n",act.data)

----------------------
Age 
 None 
 None <function DcrActivity.__init__.<locals>.<lambda> at 0x7fc94b71d800> 
 18
----------------------
Age Threshold 
 None 
 None <function DcrActivity.__init__.<locals>.<lambda> at 0x7fc94b71d760> 
 True
----------------------
Relevant laws 
 Find relevant laws about an underage child that had access to alcohol. 
 [('source', 'tool')] <bound method FindRelevantLaws.answer of <tools.find_relevant_laws.FindRelevantLaws object at 0x7fc94b724a50>> 
 The available excerpts are insufficient to identify laws governing an underage child’s access to alcohol. They contain no rules on purchasing, selling, serving, supplying, possessing, or consuming alcohol, nor any applicable age limits.

The only potentially related provision concerns the municipality’s duty when police document that a person under 18 is suspected of serious crime; it does not establish that access to alcohol constitutes such crime.[law.pdf#page=37]

Relevant provisions on alcohol sales or se

In [ ]:
t = [e.activityID for e in graph.executions]

In [ ]:
h = [graph.getActivity(x).data for x in [e.activityID for e in graph.executions]]

In [ ]:
from pm4py.objects.dcr.exporter import exporter as dcr_exporter
dcr_exporter.apply(graph,'/home/vco/Projects2026/DcrController/backend/data/models/ToolCallTest.xml'
,dcr_exporter.Variants.DCR_JS_PORTAL)

In [ ]:
from util.csvparser import CsvParser
from util.fileprocessor import FileProcessor
from util.jsonparser import JsonParser
from util.pdfparser import LocalPdfParser
from util.textparser import TextParser
from util.textsplitter import SentenceTextSplitter, SimpleTextSplitter, XmlSplitter

csv_max_chars_per_page = 1000
sentence_text_splitter = SentenceTextSplitter()
file_processors = {
    ".json": FileProcessor(JsonParser(), SimpleTextSplitter()),
    ".xml": FileProcessor(TextParser(), XmlSplitter()),
    ".md": FileProcessor(TextParser(), sentence_text_splitter),
    ".txt": FileProcessor(TextParser(), sentence_text_splitter),
    ".csv": FileProcessor(CsvParser(max_chars_per_page=csv_max_chars_per_page), sentence_text_splitter),
    ".pdf": FileProcessor(LocalPdfParser(), sentence_text_splitter),
}

In [ ]:
fp = FileProcessor(LocalPdfParser(), sentence_text_splitter)

In [ ]:
fp.parser.parse()

In [ ]:
from tools.find_relevant_dcr_graphs import FindRelevantDcrGraphs

query="Lexplain article 86"
dcr_finder = FindRelevantDcrGraphs()
for res in dcr_finder.find(query=query, top_k=2):
    print(res.score,res.format,res.source)

In [ ]:
from tools.find_similar_cases import FindSimilarCases

finder = FindSimilarCases()

results = finder.find("child disability expenses", top_k=5)
clusters = finder.cluster("child disability expenses", top_k_per_outcome=5)

print(clusters.positive)
print(clusters.negative)
print(clusters.unknown)
for result in results:
    print(result.score, result.source, result.page_number)#, result.text)

In [ ]:
from tools.find_relevant_laws import FindRelevantLaws

results = FindRelevantLaws().find("requirements for compensation", top_k=5)

for result in results:
    print(result.score, result.source, result.page_number)#, result.text)

In [ ]:
delete_response = httpx.request(
    'DELETE',
    'http://127.0.0.1:8000/api/chat/session',
    json={'session_id': execution_result['session_id']},
)
delete_response.raise_for_status()
print('Session removed from backend memory.')

In [ ]:
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
model = SentenceTransformer("google/embeddinggemma-300m")

# 2. Save to a local directory
model.save_pretrained("models/local_gemma_embedding")
print("Model successfully saved locally!")

In [ ]:
model = SentenceTransformer("models/local_gemma_embedding")
# Run inference with queries and documents
query = "Which planet is known as the Red Planet?"
documents = [
    "Venus is often called Earth's twin because of its similar size and proximity.",
    "Mars, known for its reddish appearance, is often referred to as the Red Planet.",
    "Jupiter, the largest planet in our solar system, has a prominent red spot.",
    "Saturn, famous for its rings, is sometimes mistaken for the Red Planet."
]
query_embeddings = model.encode_query(query)
document_embeddings = model.encode_document(documents)
print(query_embeddings.shape, document_embeddings.shape)
# (768,) (4, 768)

# Compute similarities to determine a ranking
similarities = model.similarity(query_embeddings, document_embeddings)
print(similarities)
# tensor([[0.3011, 0.6359, 0.4930, 0.4889]])
